# Notebook 06 — Stage 2: replicate harmonisation and duplicate checks

**Role in the pipeline:** the first stage that operates *across* rows
rather than within them. Reads Stage 1B's `analysis_ready_samples.csv`,
adds two grouping keys (`sample_month`, `depth_round_m`), checks for
duplicates, and computes per-replicate-group means + standard
deviations. Flags rows whose group has a metadata / provenance / QC /
source conflict or where the SD of a metric exceeds the GOA-ON
"weather" precision threshold.

```text
05_stage1b.ipynb
   └── <stage1b_out>/data/analysis_ready_samples.csv
                                │
                                ▼
                  06_stage2.ipynb            ◄── THIS NOTEBOOK
                     • re-resolve canonical aliases (defensive)
                     • add sample_month, depth_round_m
                     • duplicate detection (per-row + per-group summary)
                     • replicate harmonisation (means + SDs)
                     • conflict + SD-threshold flags
                     • write enhanced.csv (rows + flags) +
                       per-group tables (replicate means, conflicts, ...)
                                                  │
                                                  ▼
                                       07_stage3.ipynb → 08_stage4
```

**What Stage 2 does NOT do.** It does not rebuild `ph_best`,
`ph_co2sys`, `ta_best_umolkg`, `pco2_best_uatm`, or `dic_best_umol_kg`.
Those are Stage 1B's job and are preserved unchanged. Stage 2 only
*aggregates* and *flags*; it does not silently overwrite values.


## Parameters

Single tagged `parameters` cell. Default `INPUT_CSV` points at the new
short Stage 1B path (`<stage1b_out>/data/analysis_ready_samples.csv`),
not the original's flat `oa_stage1b_outputs\analysis_ready_samples.csv`
(which never existed in the refactored layout).


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================

# --- I/O -------------------------------------------------------------
# These are intentionally None so the notebook does not accidentally run on
# a personal local path. run_pipeline.sh supplies both values with Papermill.
INPUT_CSV = None
OUT_DIR = None

# --- Config override (optional) ------------------------------------
# Deep-merges onto oa_pipeline.stage2.STAGE2_DEFAULTS.
CONFIG_PATH = None

# --- Stage 2 behaviour ---------------------------------------------
# Rounding precision for the replicate grouping key.
DEPTH_ROUND_DECIMALS = 1

# If True, skip Parquet writes (CSV only). Parquet failures are tolerated
# and recorded in the manifest.
NO_PARQUET = False

# Useful during development: prepare everything, write nothing.
DRY_RUN = False

# Optional debugging aid.
PRINT_COLUMNS = False


## Setup

All Stage 2 logic lives in `oa_stage2.py`. Shared helpers (`die`,
`utc_stamp`, `write_json`, `write_text`, `deep_update`, `ensure_dir`,
etc.) come from `oa_common.py` — the original notebook redefined all
of them, which is the largest single duplication block in the audit.


In [ ]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

import pandas as pd

try:
    import importlib.metadata as importlib_metadata
    oa_pipeline_version = importlib_metadata.version("oa-pipeline")
except Exception:
    oa_pipeline_version = None

try:
    from IPython.display import display
except Exception:
    display = None

from oa_pipeline.common import (
    deep_update,
    die,
    ensure_dir,
    md_table_from_df,
    normalize_columns,
    utc_stamp,
    write_csv_and_parquet,
    write_json,
    write_text,
)
from oa_pipeline.schema import load_config
from oa_pipeline.stage2 import (
    STAGE2_DEFAULTS,
    add_conflict_annotations,
    add_duplicate_annotations,
    add_replicate_annotations,
    add_time_and_depth_keys,
    duplicate_check,
    ensure_required_columns,
    ensure_stage2_dirs,
    make_column_inventory,
    make_presence_table,
    materialize_canonical_aliases,
    replicate_harmonise,
)


def as_bool(value) -> bool:
    """Parse Papermill friendly boolean values safely."""
    if isinstance(value, bool):
        return value

    if value is None:
        return False

    text = str(value).strip().lower()

    if text in {"true", "1", "yes", "y"}:
        return True

    if text in {"false", "0", "no", "n", "none", "null", ""}:
        return False

    die(f"Cannot parse boolean parameter value: {value!r}")


## Load input and canonicalise

The first step *re-runs* canonical alias resolution. Why, when Stage 1B
should already have done it? Because Stage 2 should also work when fed
some other CSV that happens to have the right shape but uses different
column names. If you feed it a clean Stage 1B output the alias step is
a no-op; if you feed it something exotic it gives you the chance to
recover.


In [ ]:
NO_PARQUET = as_bool(NO_PARQUET)
DRY_RUN = as_bool(DRY_RUN)
PRINT_COLUMNS = as_bool(PRINT_COLUMNS)

try:
    DEPTH_ROUND_DECIMALS = int(DEPTH_ROUND_DECIMALS)
except Exception:
    die(f"DEPTH_ROUND_DECIMALS must be an integer, got {DEPTH_ROUND_DECIMALS!r}")

if DEPTH_ROUND_DECIMALS < 0:
    die(f"DEPTH_ROUND_DECIMALS must be >= 0, got {DEPTH_ROUND_DECIMALS}")

if INPUT_CSV is None or str(INPUT_CSV).strip() == "":
    die("INPUT_CSV is required. Run through run_pipeline.sh or set INPUT_CSV.")

if OUT_DIR is None or str(OUT_DIR).strip() == "":
    die("OUT_DIR is required. Run through run_pipeline.sh or set OUT_DIR.")

input_csv = Path(INPUT_CSV).expanduser().resolve()

if not input_csv.exists():
    die(
        f"File not found: {input_csv}\n"
        "Did Stage 1B run successfully? Stage 2 reads analysis_ready_samples.csv."
    )

if input_csv.suffix.lower() not in {".csv", ".txt"}:
    die(f"Expected a CSV-like file, got: {input_csv.name}")

config_path = None
if CONFIG_PATH is not None:
    text = str(CONFIG_PATH).strip()
    if text.lower() not in {"", "none", "null"}:
        config_path = text

user_config, config_source = load_config(config_path)

if config_path:
    config = deep_update(
        copy.deepcopy(STAGE2_DEFAULTS),
        {
            k: v
            for k, v in user_config.items()
            if k in STAGE2_DEFAULTS or k not in user_config
        },
    )
else:
    config = copy.deepcopy(STAGE2_DEFAULTS)

required_cols = config.get(
    "required_stage2_input_columns",
    config.get("required_stage2_columns", config.get("required_stage1b_columns", [])),
)
expected_cols = config.get(
    "expected_stage2_columns",
    config.get("expected_provenance_columns", []),
)

out_root = ensure_dir(Path(OUT_DIR).expanduser().resolve())
dirs = ensure_stage2_dirs(out_root)

notes: list[str] = []

df = pd.read_csv(input_csv)
df = normalize_columns(df)
df["source_file_stage2"] = str(input_csv)
df["stage2_processed_utc"] = utc_stamp()

for dcol in ["sample_date", "sample_date_dt", "date", "datetime"]:
    if dcol in df.columns:
        df[dcol] = pd.to_datetime(df[dcol], errors="coerce", utc=True)

df, alias_resolution = materialize_canonical_aliases(
    df,
    config["canonical_aliases"],
    notes,
)
ensure_required_columns(df, required_cols)

if PRINT_COLUMNS:
    print("\nColumns:")
    for col in df.columns:
        print(f"  {col}")

print(f"Rows loaded: {len(df):,}")
print(f"Columns    : {df.shape[1]}")
print(f"Output root: {out_root}")


## Canonical-field presence inventory

A one-row-per-promised-column view: required vs expected, present
vs missing, count and percent of non-NA values. Use this to verify
Stage 1B really wrote what Stage 2 needs.


In [ ]:
presence_df = make_presence_table(
    df,
    required=required_cols,
    expected=expected_cols,
)

missing_opt = presence_df.loc[
    (~presence_df["required"]) & (~presence_df["present"]), "column"
].tolist()
if missing_opt:
    notes.append("Optional expected columns not found: " + ", ".join(missing_opt))

if display is not None:
    display(presence_df)
else:
    print(presence_df.to_string(index=False))


## Add grouping helpers

Two new columns get added:

- `sample_month` — monthly period string derived from `sample_date`
  (e.g. `"2024-03"`). Used by the replicate consistency-check, not
  by the replicate-group key (the group key uses `sample_date` itself
  so that two profiles taken weeks apart at the same station stay
  apart).
- `depth_round_m` — `depth_m` rounded to `DEPTH_ROUND_DECIMALS` places.
  Used as part of the replicate-group key so casts at "essentially the
  same depth" aggregate together.

A full column inventory is generated for the audit trail.


In [ ]:
df = add_time_and_depth_keys(
    df,
    notes,
    depth_round_decimals=DEPTH_ROUND_DECIMALS,
    depth_bin_m=config.get("depth_bin_m", 1.0),
)
inventory_df = make_column_inventory(df)

if display is not None:
    display(inventory_df.head(20))
else:
    print(inventory_df.head(20).to_string(index=False))


## Duplicate checks

A "duplicate" here is any row whose `duplicate_keys` tuple
(`[sample_id, replicate_id, sample_date, station_id, depth_m]` by
default) collides with at least one other row. Two outputs:

- `dup_rows` — every row in a duplicate group (sorted by keys).
- `dup_summary` — one row per duplicate group, with min/max/range
  for each carbonate-chemistry metric. Lets an analyst spot
  same-record disagreements at a glance.

Two new columns are added to the main frame:
`flag_duplicate` and `duplicate_group_size`.


In [ ]:
dup_rows, dup_summary, dup_keys_used = duplicate_check(df, config["duplicate_keys"])
df = add_duplicate_annotations(df, dup_keys_used)

print(f"Duplicate rows  : {len(dup_rows):,}")
print(f"Duplicate groups: {len(dup_summary):,}")
print(f"Duplicate keys  : {dup_keys_used}")

if display is not None and not dup_summary.empty:
    display(dup_summary.head(20))


## Replicate harmonisation

Groups rows by `replicate_group_keys` (default:
`[cruise_id, transect_id, station_id, depth_round_m, sample_date]`) and
computes:

- **Means** of every whitelisted numeric variable
  (`replicate_mean_vars`).
- **Sample SDs** (ddof=1) of those same variables.
- **First-value** of every other column (metadata, IDs, source
  pointers...). Conflicts in those are surfaced separately in
  `consistency_df`, not silently dropped.
- **Per-class conflict flags** (metadata / qc / provenance / source /
  other) at the row level.
- **SD-threshold flag** if any metric's group SD exceeds its
  per-metric threshold. Defaults match the GOA-ON "weather"
  precision objectives (Newton et al. 2015): pH ±0.02, TA ±10 µmol/kg.

Three tables come out of this:
- `rep_mean_sd` — one row per replicate group, fully aggregated.
- `consistency_df` — one row per (group, conflicting field).
- `disagree_df` — one row per (group, metric) where the SD exceeds
  the threshold.


In [ ]:
(
    rep_mean,
    rep_mean_sd,
    consistency_df,
    disagree_df,
    rep_keys_used,
    mean_vars,
    nrep,
) = replicate_harmonise(
    df,
    requested_keys=config["replicate_group_keys"],
    mean_whitelist=config["replicate_mean_vars"],
    consistency_cols=config["replicate_consistency_check_columns"],
    sd_thresholds=config["replicate_sd_thresholds"],
    conflict_class_map=config["replicate_conflict_field_classes"],
)

df = add_replicate_annotations(df, nrep, rep_keys_used)
df = add_conflict_annotations(df, consistency_df, disagree_df, rep_keys_used)

print(f"Replicate group keys   : {rep_keys_used}")
print(f"Numeric mean variables : {len(mean_vars)}")
print(f"Replicate groups       : {len(rep_mean_sd):,}")
print(f"Consistency conflicts  : {len(consistency_df):,}")
print(f"SD threshold failures  : {len(disagree_df):,}")
print(f"SD thresholds in use   : {config['replicate_sd_thresholds']}")

if display is not None:
    if not consistency_df.empty:
        display(consistency_df.head(20))
    if not disagree_df.empty:
        display(disagree_df.head(20))


## Quick preview of the enhanced table

In [ ]:
preview_cols = [
    c for c in [
        "record_id", "sample_id", "sample_date", "station_id",
        "depth_m", "depth_round_m", "sample_month",
        "ta_best_umolkg", "ph_best", "ph_co2sys",
        "pco2_best_uatm", "dic_best_umol_kg",
        "flag_duplicate", "duplicate_group_size",
        "replicate_group_n", "flag_has_replicates",
        "flag_replicate_any_conflict", "flag_replicate_sd_exceeded",
    ]
    if c in df.columns
]

if display is not None:
    display(df[preview_cols].head(20))
else:
    print(df[preview_cols].head(20).to_string(index=False))


## Prepare output paths

Same JWST-style layout used by every other notebook in the refactor:
short filenames, identity in the parent folder. No `<stem>__` prefix
on every file (the original generated names like
`<long_input_stem>__stage2_enhanced.csv`).

```
<OUT_DIR>/
    data/
        enhanced.csv                   (and .parquet)   # ◄── Stage 3 input
    tables/
        column_inventory.csv
        canonical_presence.csv
        duplicate_rows.csv
        duplicate_summary.csv
        replicate_means_sd.csv
        replicate_consistency.csv
        replicate_disagreement.csv
    reports/
        report.md
    logs/
        manifest.json
        effective_config.json
```


In [ ]:
paths = {
    "enhanced_csv":           dirs["data"]    / "enhanced.csv",
    "enhanced_parquet":       dirs["data"]    / "enhanced.parquet",
    "column_inventory_csv":   dirs["tables"]  / "column_inventory.csv",
    "canonical_presence_csv": dirs["tables"]  / "canonical_presence.csv",
    "duplicate_rows_csv":     dirs["tables"]  / "duplicate_rows.csv",
    "duplicate_summary_csv":  dirs["tables"]  / "duplicate_summary.csv",
    "replicate_means_sd_csv": dirs["tables"]  / "replicate_means_sd.csv",
    "replicate_consistency_csv": dirs["tables"] / "replicate_consistency.csv",
    "replicate_disagreement_csv": dirs["tables"] / "replicate_disagreement.csv",
    "report_md":              dirs["reports"] / "report.md",
    "manifest_json":          dirs["logs"]    / "manifest.json",
    "effective_config_json":  dirs["logs"]    / "effective_config.json",
}
print(f"Output root: {out_root}")


## Write outputs

In [ ]:
parquet_written = False
parquet_error = None

if DRY_RUN:
    print("DRY_RUN = True -- no files written.")
else:
    # All the per-table CSVs.
    inventory_df.to_csv(paths["column_inventory_csv"], index=False)
    presence_df.to_csv(paths["canonical_presence_csv"], index=False)
    dup_rows.to_csv(paths["duplicate_rows_csv"], index=False)
    dup_summary.to_csv(paths["duplicate_summary_csv"], index=False)
    rep_mean_sd.to_csv(paths["replicate_means_sd_csv"], index=False)
    consistency_df.to_csv(paths["replicate_consistency_csv"], index=False)
    disagree_df.to_csv(paths["replicate_disagreement_csv"], index=False)

    # Main enhanced frame (row-level + flags).
    if NO_PARQUET:
        df.to_csv(paths["enhanced_csv"], index=False)
        parquet_error = "Parquet disabled by user"
    else:
        parquet_written, parquet_error = write_csv_and_parquet(
            df, paths["enhanced_csv"], paths["enhanced_parquet"]
        )

    write_json(paths["effective_config_json"], config)

    # Report
    def _flag_count(col: str) -> int:
        return int((df[col] == True).sum()) if col in df.columns else 0

    notes_md = "\n".join(f"- {n}" for n in notes) if notes else "- (none)"

    report_md_text = f"""# Stage 2 Preprocessing Report

**Generated:** {utc_stamp()}
**Input:** `{input_csv}`
**Rows:** {len(df):,}  **Columns:** {df.shape[1]:,}

## What this stage does
Validates Stage 1B canonical fields, adds grouping helpers
(`sample_month`, `sample_day`, `depth_round_m`, `depth_bin_m`), checks
duplicates, and harmonises replicates. Does **not** rebuild `ph_best`,
`ph_co2sys`, `ta_best_umolkg`, `pco2_best_uatm`, or `dic_best_umol_kg`.

## Canonical field presence
{md_table_from_df(presence_df, max_rows=200)}

## Duplicate check
- Keys used: `{dup_keys_used}`
- Duplicate rows: **{len(dup_rows):,}**
- Duplicate groups: **{len(dup_summary):,}**

## Replicate harmonisation
- Group keys used: `{rep_keys_used}`
- Variables averaged: `{mean_vars}`
- Replicate groups: **{len(rep_mean_sd):,}**
- Consistency conflicts: **{len(consistency_df):,}**
- SD-threshold failures: **{len(disagree_df):,}**
- SD thresholds used: `{config['replicate_sd_thresholds']}`

## Conflict / SD flags (row counts)
| Flag | Rows |
|------|-----:|
| metadata conflict | {_flag_count("flag_replicate_metadata_conflict"):,} |
| provenance conflict | {_flag_count("flag_replicate_provenance_conflict"):,} |
| qc conflict | {_flag_count("flag_replicate_qc_conflict"):,} |
| source conflict | {_flag_count("flag_replicate_source_conflict"):,} |
| other conflict | {_flag_count("flag_replicate_other_conflict"):,} |
| any conflict | {_flag_count("flag_replicate_any_conflict"):,} |
| SD threshold exceeded | {_flag_count("flag_replicate_sd_exceeded"):,} |

## Column inventory (top 30 by missingness)
{md_table_from_df(inventory_df.head(30), max_rows=200)}

## Reference context
The replicate SD warning thresholds are based on first pass GOA ON
weather quality targets commonly used for ocean acidification observing:
approximately ±0.02 pH units and ±10 µmol kg⁻¹ for total alkalinity or DIC.
These thresholds are used here as screening thresholds, not as formal
uncertainty estimates.

## Notes
{notes_md}

## Main outputs
- Enhanced CSV : `{paths["enhanced_csv"]}`  (Stage 3 reads this)
- Replicate means + SD: `{paths["replicate_means_sd_csv"]}`
"""
    write_text(paths["report_md"], report_md_text)

    manifest = {
        "notebook": "06_stage2",
        "generated_utc": utc_stamp(),
        "input_csv": str(input_csv),
        "output_root": str(out_root),
        "config_source": config_source,
        "parameters": {
            "INPUT_CSV": str(input_csv),
            "OUT_DIR": str(out_root),
            "CONFIG_PATH": CONFIG_PATH,
            "config_path_resolved": config_path,
            "DEPTH_ROUND_DECIMALS": DEPTH_ROUND_DECIMALS,
            "NO_PARQUET": NO_PARQUET,
            "DRY_RUN": DRY_RUN,
        },
        "alias_resolution": alias_resolution,
        "duplicate_keys_used": dup_keys_used,
        "replicate_group_keys_used": rep_keys_used,
        "replicate_mean_vars_used": mean_vars,
        "replicate_sd_thresholds": config["replicate_sd_thresholds"],
        "row_counts": {
            "rows_loaded": int(len(df)),
            "duplicate_rows": int(len(dup_rows)),
            "duplicate_groups": int(len(dup_summary)),
            "replicate_groups": int(len(rep_mean_sd)),
            "consistency_conflicts": int(len(consistency_df)),
            "sd_failures": int(len(disagree_df)),
        },
        "flag_counts": {
            "flag_duplicate": _flag_count("flag_duplicate"),
            "flag_has_replicates": _flag_count("flag_has_replicates"),
            "flag_replicate_metadata_conflict": _flag_count("flag_replicate_metadata_conflict"),
            "flag_replicate_provenance_conflict": _flag_count("flag_replicate_provenance_conflict"),
            "flag_replicate_qc_conflict": _flag_count("flag_replicate_qc_conflict"),
            "flag_replicate_source_conflict": _flag_count("flag_replicate_source_conflict"),
            "flag_replicate_other_conflict": _flag_count("flag_replicate_other_conflict"),
            "flag_replicate_any_conflict": _flag_count("flag_replicate_any_conflict"),
            "flag_replicate_sd_exceeded": _flag_count("flag_replicate_sd_exceeded"),
        },
        "parquet_written": parquet_written,
        "parquet_error": parquet_error,
        "notes": notes,
        "outputs": {k: str(v) for k, v in paths.items()},
        "package_versions": {
            "python": sys.version.split()[0],
            "pandas": pd.__version__,
            "oa_pipeline": oa_pipeline_version,
        },
    }
    write_json(paths["manifest_json"], manifest)

    print("\nStage 2 complete.")
    print(f"  -> Stage 3 input: {paths['enhanced_csv']}")


## Review written outputs

In [ ]:
if not DRY_RUN:
    outputs_df = pd.DataFrame(
        {"output_name": list(paths.keys()), "path": [str(p) for p in paths.values()]}
    )
    if display is not None:
        display(outputs_df)
    else:
        print(outputs_df.to_string(index=False))
